In [1]:
import pandas as pd
import random

# === Load your chunked wiki dataset ===
wiki_path = r"C:\Users\choud\Downloads\wiki_hybrid_chunks_300.csv"
wiki_df = pd.read_csv(wiki_path)

# === Synthetic QA-Feedback Generator ===
def generate_synthetic_feedback(wiki_df, n=200):
    feedback_entries = []

    for _ in range(n):
        row = wiki_df.sample(1).iloc[0]
        chunk_text = row['chunk_text']
        chunk_id = row['chunk_id']

        # Extract a fake "entity" and make a query
        words = chunk_text.split()
        if len(words) < 10:
            continue

        entity = words[0].capitalize()
        question_type = random.choice(["is", "was", "did", "worked as", "appeared in"])
        query = f"What {question_type} {entity} do?"

        # Simulate answer quality
        feedback_type = random.choices(
            ["good", "partially correct", "incorrect", "hallucinated"],
            weights=[0.5, 0.2, 0.2, 0.1],
            k=1
        )[0]

        # Build answers based on feedback type
        if feedback_type == "good":
            answer = " ".join(words[:30]) + "..."
        elif feedback_type == "partially correct":
            answer = " ".join(words[:15]) + " [some details missing]..."
        elif feedback_type == "incorrect":
            answer = "This chunk refers to something else entirely."
        elif feedback_type == "hallucinated":
            answer = f"{entity} was a spaceship pilot in World War II."

        feedback_entries.append({
            "query": query,
            "answer": answer,
            "source": chunk_id,
            "feedback": feedback_type
        })

    return pd.DataFrame(feedback_entries)

# === Generate the synthetic dataset ===
synthetic_feedback_df = generate_synthetic_feedback(wiki_df, n=200)

# === Save to CSV ===
output_path = "synthetic_feedback_dataset.csv"
synthetic_feedback_df.to_csv(output_path, index=False)

print(f"✅ Done. Saved to {output_path}")


✅ Done. Saved to synthetic_feedback_dataset.csv


In [2]:
import pandas as pd
import random

# Load your labeled feedback file
df = pd.read_csv(r"C:\Users\choud\synthetic_feedback_dataset.csv")  # Replace with your actual file

# Step 1: Group answers by the same query
grouped = df.groupby("query")

pairwise_data = []

for query, group in grouped:
    # Split answers by feedback quality
    good_or_better = group[group['feedback'].isin(['good'])]
    worse = group[group['feedback'].isin(['partially correct', 'hallucinated', 'incorrect'])]

    # Generate (chosen, rejected) pairs
    for _, good_row in good_or_better.iterrows():
        for _, bad_row in worse.iterrows():
            pairwise_data.append({
                "query": query,
                "chosen_answer": good_row["answer"],
                "rejected_answer": bad_row["answer"]
            })

# Shuffle for randomness
random.shuffle(pairwise_data)

# Convert to DataFrame
pairwise_df = pd.DataFrame(pairwise_data)

# Save to CSV
pairwise_df.to_csv("pairwise_feedback_for_reward_model.csv", index=False)

print(f"✅ Created {len(pairwise_df)} pairwise comparisons.")


✅ Created 27 pairwise comparisons.


In [3]:
pip install trl transformers datasets peft accelerate


Note: you may need to restart the kernel to use updated packages.
